# 🤖 Module 6 Lab — LoanBot v2.0 Multi-Agent Architecture

**FinTech Corp AI Engineering Lab — Final Project**

---

Đây là lab cuối cùng của chương trình. Bạn sẽ xây dựng **LoanBot v2.0** — hệ thống multi-agent với Supervisor Pattern, parallelism, distributed tracing và MCP-inspired tool routing.

## Mục tiêu
1. **Agent Framework** — base classes cho multi-agent communication
2. **Specialized Agents** — CreditAgent, IncomeAgent, RiskAgent, ReportAgent
3. **Supervisor** — orchestrate parallel + sequential execution
4. **Distributed Tracer** — trace_id propagation, flamegraph
5. **LoanBot v2.0 Integration** — full system test
6. **Performance Comparison** — v1.0 vs v2.0

```
pip install anthropic
```

---
## Part 1: Agent Framework

Base classes cho multi-agent communication và message passing.

In [ ]:
import asyncio
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Callable
from enum import Enum

class MessageType(Enum):
    TASK    = "TASK"    # Supervisor → Agent: here's what to do
    RESULT  = "RESULT"  # Agent → Supervisor: here's what I found
    ERROR   = "ERROR"   # Agent → Supervisor: something went wrong
    CLARIFY = "CLARIFY" # Agent → Supervisor: need more info

class MessageStatus(Enum):
    SUCCESS = "SUCCESS"
    PARTIAL = "PARTIAL"
    FAILED  = "FAILED"
    PENDING = "PENDING"

@dataclass
class AgentMessage:
    from_agent: str
    to_agent: str
    message_type: MessageType
    payload: Dict[str, Any]
    trace_id: str = field(default_factory=lambda: f"TRC-{uuid.uuid4().hex[:8].upper()}")
    message_id: str = field(default_factory=lambda: f"MSG-{uuid.uuid4().hex[:6].upper()}")
    status: MessageStatus = MessageStatus.PENDING
    timestamp: str = field(default_factory=lambda: time.strftime("%Y-%m-%dT%H:%M:%SZ"))
    span_id: str = field(default_factory=lambda: f"SPN-{uuid.uuid4().hex[:6].upper()}")
    parent_span_id: Optional[str] = None

@dataclass
class Span:
    span_id: str
    agent_name: str
    operation: str
    start_time: float
    end_time: Optional[float] = None
    status: str = "running"
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    @property
    def duration_s(self) -> Optional[float]:
        if self.end_time and self.start_time:
            return round(self.end_time - self.start_time, 2)
        return None

class DistributedTracer:
    """Collects spans from all agents for a given trace."""
    
    def __init__(self, trace_id: str):
        self.trace_id = trace_id
        self.spans: List[Span] = []
        self._start_time = time.time()
    
    def start_span(self, agent_name: str, operation: str, metadata: dict = None) -> Span:
        span = Span(
            span_id=f"SPN-{agent_name[:4].upper()}-{uuid.uuid4().hex[:4].upper()}",
            agent_name=agent_name,
            operation=operation,
            start_time=time.time(),
            metadata=metadata or {}
        )
        self.spans.append(span)
        return span
    
    def end_span(self, span: Span, status: str = "ok", metadata: dict = None):
        span.end_time = time.time()
        span.status = status
        if metadata:
            span.metadata.update(metadata)
    
    def print_flamegraph(self):
        print(f"\n{'='*65}")
        print(f"  Distributed Trace: {self.trace_id}")
        print(f"  Total elapsed: {time.time() - self._start_time:.1f}s")
        print(f"{'='*65}")
        print(f"  {'Agent':<20} {'Operation':<20} {'Duration':>10} {'Status':>10}")
        print(f"  {'─'*62}")
        total = time.time() - self._start_time
        for span in sorted(self.spans, key=lambda s: s.start_time):
            dur = span.duration_s
            # ASCII bar
            pct = (dur / total) * 30 if dur else 0
            bar = "█" * int(pct)
            dur_str = f"{dur:.1f}s" if dur else "running"
            status_icon = "✅" if span.status == "ok" else "❌" if span.status == "error" else "⏳"
            print(f"  {span.agent_name:<20} {span.operation:<20} {dur_str:>10} {status_icon:>10}  {bar}")
        print(f"{'='*65}\n")

class BaseAgent:
    """Base class for all LoanBot v2.0 agents."""
    
    def __init__(self, agent_id: str):
        self.agent_id = agent_id
    
    async def run(self, task: AgentMessage, tracer: DistributedTracer) -> AgentMessage:
        raise NotImplementedError
    
    def _make_result(self, task: AgentMessage, payload: dict,
                     status: MessageStatus = MessageStatus.SUCCESS) -> AgentMessage:
        return AgentMessage(
            from_agent=self.agent_id,
            to_agent=task.from_agent,
            message_type=MessageType.RESULT,
            payload=payload,
            trace_id=task.trace_id,
            status=status
        )

print("✅ Agent Framework loaded")

---
## Part 2: Specialized Agents

Implement 4 specialized agents: CreditAgent, IncomeAgent, RiskAgent, ReportAgent.

In [ ]:
# Mock data store (simulating CIC, income DB, etc.)
MOCK_DB = {
    "FC-2024-001": {"credit_score": 720, "income_verified": 35_000_000, "blacklisted": False, "name": "Nguyễn Văn An"},
    "FC-2024-002": {"credit_score": 580, "income_verified": 12_000_000, "blacklisted": False, "name": "Trần Thị Bình"},
    "FC-2024-003": {"credit_score": 400, "income_verified": 8_000_000,  "blacklisted": True,  "name": "Lê Văn Cường"},
    "FC-2024-004": {"credit_score": 645, "income_verified": 20_000_000, "blacklisted": False, "name": "Phạm Thị Dung"},  # borderline
}

class CreditAgent(BaseAgent):
    """Checks credit score and blacklist status via CIC."""
    
    def __init__(self):
        super().__init__("CreditAgent")
    
    async def run(self, task: AgentMessage, tracer: DistributedTracer) -> AgentMessage:
        span = tracer.start_span(self.agent_id, "credit_check")
        await asyncio.sleep(0.3)  # Simulate CIC API latency
        
        customer_id = task.payload["customer_id"]
        data = MOCK_DB.get(customer_id)
        
        if not data:
            tracer.end_span(span, "error")
            return self._make_result(task, {"error": f"Customer {customer_id} not found"},
                                     MessageStatus.FAILED)
        
        payload = {
            "customer_id": customer_id,
            "credit_score": data["credit_score"],
            "credit_grade": "A" if data["credit_score"] >= 700 else "B" if data["credit_score"] >= 600 else "C",
            "blacklisted": data["blacklisted"],
            "data_source": "CIC API v3"
        }
        tracer.end_span(span, "ok", {"credit_score": data["credit_score"]})
        print(f"  [{self.agent_id}] score={data['credit_score']}, blacklisted={data['blacklisted']}")
        return self._make_result(task, payload)


class IncomeAgent(BaseAgent):
    """Verifies income and calculates DTI ratio."""
    
    def __init__(self):
        super().__init__("IncomeAgent")
    
    async def run(self, task: AgentMessage, tracer: DistributedTracer) -> AgentMessage:
        span = tracer.start_span(self.agent_id, "income_verify")
        await asyncio.sleep(0.4)  # Simulate income verification (slightly slower than credit)
        
        p = task.payload
        customer_id = p["customer_id"]
        data = MOCK_DB.get(customer_id, {})
        verified_income = data.get("income_verified", 0)
        
        # Calculate monthly payment: P × r / (1 - (1+r)^-n)
        loan_amount = p["loan_amount"]
        annual_rate = p.get("annual_rate", 0.10)
        term_months = p.get("term_months", 36)
        monthly_rate = annual_rate / 12
        if monthly_rate > 0:
            monthly_payment = loan_amount * monthly_rate / (1 - (1 + monthly_rate) ** -term_months)
        else:
            monthly_payment = loan_amount / term_months
        
        existing_debt = p.get("existing_debt", 0)
        dti = (existing_debt + monthly_payment) / verified_income if verified_income > 0 else 1.0
        
        payload = {
            "customer_id": customer_id,
            "declared_income": p.get("declared_income", 0),
            "verified_income": verified_income,
            "monthly_payment": round(monthly_payment, 0),
            "dti_ratio": round(dti, 3),
            "is_dti_acceptable": dti <= 0.43
        }
        tracer.end_span(span, "ok", {"dti": round(dti, 3)})
        print(f"  [{self.agent_id}] dti={dti:.1%}, monthly_payment={monthly_payment:,.0f} VNĐ")
        return self._make_result(task, payload)


class RiskAgent(BaseAgent):
    """Aggregates credit + income signals, computes composite risk score."""
    
    def __init__(self):
        super().__init__("RiskAgent")
    
    async def run(self, task: AgentMessage, tracer: DistributedTracer) -> AgentMessage:
        span = tracer.start_span(self.agent_id, "risk_assessment")
        await asyncio.sleep(0.2)  # Simulated reasoning time
        
        credit = task.payload["credit_result"]
        income = task.payload["income_result"]
        
        # Determine decision
        if credit["blacklisted"]:
            decision, reason, risk_level = "REJECTED", "Customer is blacklisted", "CRITICAL"
        elif credit["credit_score"] >= 650 and income["is_dti_acceptable"]:
            decision, reason, risk_level = "APPROVED", f"Credit {credit['credit_score']} ≥ 650, DTI {income['dti_ratio']:.1%} ≤ 43%", "LOW"
        elif (600 <= credit["credit_score"] < 650) or (0.43 < income["dti_ratio"] <= 0.50):
            decision, reason, risk_level = "PENDING_REVIEW", f"Borderline: score={credit['credit_score']}, DTI={income['dti_ratio']:.1%}", "MEDIUM"
        else:
            decision = "REJECTED"
            reason = f"Credit {credit['credit_score']} < 650" if credit["credit_score"] < 580 else f"DTI {income['dti_ratio']:.1%} > 50%"
            risk_level = "HIGH"
        
        # Composite risk score (0.0-1.0, lower = less risky)
        credit_component = max(0, (850 - credit["credit_score"]) / 550) * 0.5
        dti_component = min(1, income["dti_ratio"] / 0.6) * 0.5
        risk_score = round(credit_component + dti_component, 3)
        
        payload = {
            "decision": decision,
            "reason": reason,
            "risk_level": risk_level,
            "risk_score": risk_score,
            "requires_hitl": task.payload.get("loan_amount", 0) > 500_000_000 or decision == "PENDING_REVIEW"
        }
        tracer.end_span(span, "ok", {"decision": decision, "risk_score": risk_score})
        print(f"  [{self.agent_id}] decision={decision}, risk_score={risk_score}, requires_hitl={payload['requires_hitl']}")
        return self._make_result(task, payload)


class ReportAgent(BaseAgent):
    """Generates human-readable decision report."""
    
    def __init__(self):
        super().__init__("ReportAgent")
    
    async def run(self, task: AgentMessage, tracer: DistributedTracer) -> AgentMessage:
        span = tracer.start_span(self.agent_id, "generate_report")
        await asyncio.sleep(0.15)  # Report generation
        
        p = task.payload
        risk = p["risk_result"]
        credit = p["credit_result"]
        income = p["income_result"]
        customer_id = p["customer_id"]
        
        report_id = f"RPT-{customer_id}-{uuid.uuid4().hex[:6].upper()}"
        
        decision_icon = {"APPROVED": "✅", "REJECTED": "❌", "PENDING_REVIEW": "⏳"}.get(risk["decision"], "?")
        
        report_text = f"""
╔══════════════════════════════════════╗
║  LOAN DECISION REPORT — {report_id}  
╠══════════════════════════════════════╣
║ Customer:    {customer_id}
║ Decision:    {decision_icon} {risk['decision']}
║ Risk Level:  {risk['risk_level']}
║ Reason:      {risk['reason']}
╠══════════════════════════════════════╣
║ Credit Score:  {credit['credit_score']} (Grade {credit['credit_grade']})
║ DTI Ratio:     {income['dti_ratio']:.1%}
║ Monthly Pmnt:  {income['monthly_payment']:,.0f} VNĐ
║ Risk Score:    {risk['risk_score']:.3f} (0=safest, 1=riskiest)
╠══════════════════════════════════════╣
║ Human Review:  {'REQUIRED' if risk['requires_hitl'] else 'NOT REQUIRED'}
╚══════════════════════════════════════╝"""
        
        payload = {
            "report_id": report_id,
            "customer_id": customer_id,
            "decision": risk["decision"],
            "report_text": report_text,
            "requires_hitl": risk["requires_hitl"]
        }
        tracer.end_span(span, "ok", {"report_id": report_id})
        return self._make_result(task, payload)

print("✅ Specialized Agents loaded: CreditAgent, IncomeAgent, RiskAgent, ReportAgent")

---
## Part 3: Supervisor — Parallel Orchestration

In [ ]:
class LoanBotSupervisor:
    """Orchestrates 4 specialized agents with parallel execution."""
    
    def __init__(self):
        self.agent_id = "Supervisor"
        self.credit_agent = CreditAgent()
        self.income_agent = IncomeAgent()
        self.risk_agent   = RiskAgent()
        self.report_agent = ReportAgent()
    
    async def process_loan(self, loan_request: dict) -> dict:
        trace_id = f"TRC-V2-{uuid.uuid4().hex[:8].upper()}"
        tracer = DistributedTracer(trace_id)
        
        print(f"\n🤖 LoanBot v2.0 Supervisor | trace_id={trace_id}")
        print(f"   Request: {loan_request['customer_id']}, loan={loan_request.get('loan_amount',0)/1e6:.0f}M VNĐ")
        print(f"   {'─'*55}")
        
        sup_span = tracer.start_span("Supervisor", "orchestrate")
        start_time = time.time()
        
        # Step 1: Create tasks for CreditAgent and IncomeAgent (run in PARALLEL)
        credit_task = AgentMessage(
            from_agent="Supervisor", to_agent="CreditAgent",
            message_type=MessageType.TASK,
            payload={"customer_id": loan_request["customer_id"],
                     "national_id": loan_request.get("national_id", "")},
            trace_id=trace_id
        )
        income_task = AgentMessage(
            from_agent="Supervisor", to_agent="IncomeAgent",
            message_type=MessageType.TASK,
            payload={
                "customer_id": loan_request["customer_id"],
                "declared_income": loan_request.get("declared_income", 0),
                "existing_debt":   loan_request.get("existing_debt", 0),
                "loan_amount":     loan_request["loan_amount"],
                "annual_rate":     loan_request.get("annual_rate", 0.10),
                "term_months":     loan_request.get("term_months", 36),
            },
            trace_id=trace_id
        )
        
        # PARALLEL EXECUTION
        print(f"   ⚡ Dispatching CreditAgent + IncomeAgent in PARALLEL...")
        credit_result, income_result = await asyncio.gather(
            self.credit_agent.run(credit_task, tracer),
            self.income_agent.run(income_task, tracer)
        )
        
        # Check for failures
        if credit_result.status == MessageStatus.FAILED:
            tracer.end_span(sup_span, "error")
            return {"error": credit_result.payload.get("error"), "trace_id": trace_id}
        
        parallel_time = time.time() - start_time
        print(f"   ✅ Parallel phase complete in {parallel_time:.2f}s")
        
        # Step 2: RiskAgent (sequential, needs both results)
        risk_task = AgentMessage(
            from_agent="Supervisor", to_agent="RiskAgent",
            message_type=MessageType.TASK,
            payload={
                "credit_result": credit_result.payload,
                "income_result": income_result.payload,
                "loan_amount": loan_request["loan_amount"]
            },
            trace_id=trace_id
        )
        risk_result = await self.risk_agent.run(risk_task, tracer)
        
        # Step 3: ReportAgent
        report_task = AgentMessage(
            from_agent="Supervisor", to_agent="ReportAgent",
            message_type=MessageType.TASK,
            payload={
                "customer_id":   loan_request["customer_id"],
                "credit_result": credit_result.payload,
                "income_result": income_result.payload,
                "risk_result":   risk_result.payload
            },
            trace_id=trace_id
        )
        report_result = await self.report_agent.run(report_task, tracer)
        
        tracer.end_span(sup_span, "ok")
        total_time = time.time() - start_time
        
        # Print report
        print(report_result.payload["report_text"])
        print(f"   Total latency: {total_time:.2f}s")
        
        # Print trace
        tracer.print_flamegraph()
        
        return {
            "trace_id": trace_id,
            "decision": risk_result.payload["decision"],
            "report_id": report_result.payload["report_id"],
            "total_latency_s": total_time,
            "requires_hitl": risk_result.payload["requires_hitl"]
        }

print("✅ LoanBot v2.0 Supervisor ready")

---
## Part 4: Full System Test

Test LoanBot v2.0 với 4 loan cases từ FinTech Corp.

In [ ]:
supervisor = LoanBotSupervisor()

test_cases = [
    {
        "customer_id": "FC-2024-001",
        "national_id": "001234567890",
        "declared_income": 35_000_000,
        "existing_debt":    5_000_000,
        "loan_amount":    300_000_000,
        "annual_rate":          0.10,
        "term_months":            36,
    },
    {
        "customer_id": "FC-2024-003",  # Blacklisted
        "national_id": "003234567890",
        "declared_income": 8_000_000,
        "existing_debt":   2_000_000,
        "loan_amount":   100_000_000,
        "annual_rate":         0.10,
        "term_months":           24,
    },
    {
        "customer_id": "FC-2024-004",  # Borderline
        "national_id": "004234567890",
        "declared_income": 20_000_000,
        "existing_debt":    6_000_000,
        "loan_amount":    400_000_000,
        "annual_rate":          0.10,
        "term_months":            60,
    },
]

# Run all test cases
async def run_all_tests():
    results = []
    for tc in test_cases:
        result = await supervisor.process_loan(tc)
        results.append(result)
    return results

results = asyncio.run(run_all_tests())

print("\n" + "="*65)
print("  LoanBot v2.0 — Test Summary")
print("="*65)
for i, r in enumerate(results):
    print(f"  Case {i+1}: {r.get('decision','ERROR')} | {r.get('total_latency_s',0):.2f}s | HITL={'YES' if r.get('requires_hitl') else 'NO'}")

---
## Part 5: Performance Comparison v1.0 vs v2.0

In [ ]:
# Simulate throughput comparison
import statistics

# v1.0: single agent, sequential (simulated)
V1_LATENCIES = [1.2, 1.4, 1.1, 1.6, 1.3, 1.5, 1.2, 1.8, 1.3, 1.1]  # seconds (mock)
# v2.0: multi-agent, parallel (our actual results + some noise)
V2_LATENCIES = [r["total_latency_s"] for r in results if "total_latency_s" in r]
# Pad with simulated for more data
V2_LATENCIES += [0.85, 0.92, 0.78, 0.96, 0.83, 0.88, 0.91]

def percentile(data, p):
    sorted_data = sorted(data)
    idx = int(len(sorted_data) * p / 100)
    return sorted_data[min(idx, len(sorted_data)-1)]

print("\n" + "="*65)
print("  Performance Comparison: LoanBot v1.0 vs v2.0")
print("="*65)
print(f"\n  {'Metric':<25} {'v1.0':>12} {'v2.0':>12} {'Improvement':>12}")
print(f"  {'─'*60}")

metrics = [
    ("Mean latency (s)", statistics.mean(V1_LATENCIES), statistics.mean(V2_LATENCIES)),
    ("P95 latency (s)", percentile(V1_LATENCIES, 95), percentile(V2_LATENCIES, 95)),
    ("P50 latency (s)", percentile(V1_LATENCIES, 50), percentile(V2_LATENCIES, 50)),
]

for name, v1, v2 in metrics:
    pct_improvement = (v1 - v2) / v1 * 100
    print(f"  {name:<25} {v1:>12.2f} {v2:>12.2f} {pct_improvement:>10.1f}%")

# Throughput
v1_throughput = 3600 / statistics.mean(V1_LATENCIES)
v2_throughput = 3600 / statistics.mean(V2_LATENCIES)
print(f"  {'Throughput (req/hr)':>25} {v1_throughput:>12.0f} {v2_throughput:>12.0f} {(v2_throughput/v1_throughput-1)*100:>10.0f}%")

# Cost (estimated)
v1_cost = 0.005  # USD (all-haiku, sequential)
v2_cost = 0.018  # USD (mixed tiering)
print(f"  {'Est. cost/request':>25} ${v1_cost:>10.4f} ${v2_cost:>10.4f} {'More expensive':>12}")

print(f"""
  Summary:
  - v2.0 is ~{(v2_throughput/v1_throughput):.1f}x faster throughput than v1.0
  - v2.0 costs ~{v2_cost/v1_cost:.1f}x more per request (mixed model tiering)
  - For 200K requests/month: v2.0 cost = ${200000*v2_cost:,.0f}/month
  - At scale, throughput gain > cost increase for FinTech Corp use case
""")

---
## Part 6: Real Claude API — Supervisor with LLM Intelligence

Tích hợp Claude API thực tế cho Supervisor để có intelligent orchestration.

In [ ]:
import os

try:
    import anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False

class IntelligentSupervisor(LoanBotSupervisor):
    """
    LoanBot v2.0 Supervisor with Claude as the decision-making brain.
    Uses Claude to evaluate aggregated results and explain decisions.
    """
    
    SUPERVISOR_SYSTEM = """
Bạn là Supervisor Agent của LoanBot v2.0 — hệ thống thẩm định tín dụng của FinTech Corp.

Bạn nhận kết quả từ 3 specialized agents:
- CreditAgent: credit score, blacklist status
- IncomeAgent: verified income, DTI ratio
- RiskAgent: risk score, recommended decision

Nhiệm vụ: Đánh giá tổng hợp và đưa ra final decision với explanation rõ ràng.

Tiêu chuẩn:
- APPROVED: credit ≥ 650 AND DTI ≤ 43%
- PENDING_REVIEW: borderline hoặc HITL required
- REJECTED: blacklisted, credit < 580, hoặc DTI > 50%

QUAN TRỌNG: Chỉ dùng số liệu từ agent results. Không bịa."""
    
    async def process_loan_with_llm(self, loan_request: dict) -> dict:
        if not ANTHROPIC_AVAILABLE or not os.environ.get("ANTHROPIC_API_KEY"):
            print("⚠️  Using mock Supervisor (no API key). Running standard v2.0...")
            return await self.process_loan(loan_request)
        
        trace_id = f"TRC-V2-LLM-{uuid.uuid4().hex[:6].upper()}"
        tracer = DistributedTracer(trace_id)
        
        print(f"\n🤖 Intelligent LoanBot v2.0 | trace_id={trace_id}")
        
        # Step 1: Parallel CreditAgent + IncomeAgent
        credit_task = AgentMessage(
            from_agent="Supervisor", to_agent="CreditAgent",
            message_type=MessageType.TASK,
            payload={"customer_id": loan_request["customer_id"], "national_id": ""},
            trace_id=trace_id
        )
        income_task = AgentMessage(
            from_agent="Supervisor", to_agent="IncomeAgent",
            message_type=MessageType.TASK,
            payload={
                "customer_id": loan_request["customer_id"],
                "declared_income": loan_request.get("declared_income", 0),
                "existing_debt":   loan_request.get("existing_debt", 0),
                "loan_amount":     loan_request["loan_amount"],
                "annual_rate":     0.10, "term_months": 36,
            },
            trace_id=trace_id
        )
        
        print("  ⚡ Parallel: CreditAgent + IncomeAgent...")
        credit_res, income_res = await asyncio.gather(
            self.credit_agent.run(credit_task, tracer),
            self.income_agent.run(income_task, tracer)
        )
        
        # Step 2: Claude as Supervisor brain
        sup_span = tracer.start_span("Supervisor-LLM", "final_decision")
        client = anthropic.Anthropic()
        summary = f"""Kết quả từ các agents:
CreditAgent: {json.dumps(credit_res.payload)}
IncomeAgent: {json.dumps(income_res.payload)}
Khoản vay yêu cầu: {loan_request['loan_amount']:,} VNĐ

Hãy đánh giá và đưa ra final decision với explanation ngắn gọn cho khách hàng."""
        
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            system=self.SUPERVISOR_SYSTEM,
            messages=[{"role": "user", "content": summary}]
        )
        llm_decision = resp.content[0].text
        tracer.end_span(sup_span, "ok")
        
        print(f"\n📋 Supervisor LLM Decision:\n{llm_decision}")
        tracer.print_flamegraph()
        
        return {"trace_id": trace_id, "llm_decision": llm_decision}

# Test with real API
isup = IntelligentSupervisor()
test_case = {
    "customer_id": "FC-2024-001",
    "declared_income": 35_000_000,
    "existing_debt":    5_000_000,
    "loan_amount":    300_000_000,
}
result = asyncio.run(isup.process_loan_with_llm(test_case))
print(f"\nFinal result: trace_id={result.get('trace_id')}")

---
## 💡 Exercises

### Exercise 1: Debate Pattern
Implement Debate Pattern cho borderline cases: spawn 3 RiskAgents với slightly different parameters, take majority vote.

In [ ]:
class DebateRiskAgent(RiskAgent):
    def __init__(self, agent_id: str, dti_threshold: float = 0.43):
        super().__init__()
        self.agent_id = agent_id
        self.dti_threshold = dti_threshold  # slightly different threshold per agent

async def debate_decision(credit_payload: dict, income_payload: dict, loan_amount: float) -> str:
    """
    Implement Debate Pattern: 3 RiskAgents with different DTI thresholds vote.
    Return majority decision.
    """
    tracer = DistributedTracer("TRC-DEBATE")
    
    # TODO: Create 3 DebateRiskAgents with thresholds 0.40, 0.43, 0.46
    # Run all 3 in parallel with asyncio.gather
    # Count votes: APPROVED vs REJECTED vs PENDING_REVIEW
    # Return majority decision
    
    return "TODO"

# Test with FC-2024-004 (borderline)
# credit_payload = {"credit_score": 645, "credit_grade": "B", "blacklisted": False}
# income_payload = {"dti_ratio": 0.44, "is_dti_acceptable": False, ...}
print("TODO: Implement debate_decision() for Exercise 1")

### Exercise 2: MCP Tool Registry
Implement a simplified MCP Server that registers tools and handles authorization.

In [ ]:
from typing import Callable

class MCPServer:
    """Simplified MCP Server for LoanBot v2.0."""
    
    def __init__(self, server_name: str):
        self.server_name = server_name
        self.tools = {}
        self.permissions = {}  # agent_id → list of allowed tools
    
    def register_tool(self, name: str, fn: Callable, description: str, allowed_agents: list):
        # TODO: store tool and permissions
        pass
    
    def list_tools(self, agent_id: str) -> list:
        # TODO: return list of tools agent is allowed to use
        return []
    
    def call_tool(self, agent_id: str, tool_name: str, params: dict) -> dict:
        # TODO: check permission, execute tool, return result
        return {"error": "TODO"}

# Test MCPServer
cic_mcp = MCPServer("CIC-MCP")
# cic_mcp.register_tool("check_credit_score", lambda p: {"score": 720}, "CIC credit", ["CreditAgent"])
# cic_mcp.register_tool("check_blacklist",    lambda p: {"blacklisted": False}, "Blacklist", ["CreditAgent"])

# Test: CreditAgent should access, ReportAgent should NOT
# print(cic_mcp.call_tool("CreditAgent",  "check_credit_score", {"customer_id": "FC-001"}))  # OK
# print(cic_mcp.call_tool("ReportAgent",  "check_credit_score", {"customer_id": "FC-001"}))  # 403
print("TODO: Implement MCPServer for Exercise 2")

### Exercise 3: Throughput Benchmark
Simulate processing 10 loan requests with v2.0 concurrently and measure actual throughput.

In [ ]:
async def batch_process(requests: list) -> dict:
    """
    Process multiple loan requests concurrently.
    Returns: {total_time, requests_processed, throughput_per_hour}
    """
    # TODO: use asyncio.gather to process all requests concurrently
    # Measure total wall-clock time
    # Calculate throughput = len(requests) / total_time * 3600
    pass

# Generate 10 test requests
batch_requests = [
    {"customer_id": f"FC-{2024+i:04d}", "declared_income": 25_000_000 + i*5_000_000,
     "existing_debt": 3_000_000, "loan_amount": 200_000_000 + i*50_000_000}
    for i in range(5)
]

# result = asyncio.run(batch_process(batch_requests))
print("TODO: Implement batch_process() for Exercise 3")
print("Expected: ~5x throughput vs sequential processing")

---
## 🎓 Final Module Summary

Trong lab cuối này, bạn đã build hoàn chỉnh **LoanBot v2.0** với:

| Component | Implementation |
|-----------|----------------|
| Agent Framework | AgentMessage, DistributedTracer, BaseAgent |
| CreditAgent | CIC check + blacklist, async |
| IncomeAgent | Income verify + DTI calc, async |
| RiskAgent | Composite risk score, decision logic |
| ReportAgent | Formatted decision report |
| Supervisor | asyncio.gather parallel, sequential orchestration |
| Distributed Tracing | Spans, flamegraph visualization |
| LLM Integration | Claude as intelligent Supervisor brain |

### LoanBot Evolution (6 Modules)

| Version | Module | Key Additions |
|---------|--------|---------------|
| v0.1 | M1 | Basic Claude API, 5 tools |
| v0.2 | M2 | CircuitBreaker, MemoryManager, HITL, AuditLog |
| v0.3 | M3 | Eval suite, 12 metrics, F1=95.5% |
| v1.0 | M4 | Docker, CI/CD Quality Gate, Canary deploy |
| v1.1 | M5 | EU AI Act, Security Pipeline, Fairness Audit |
| v2.0 | M6 | Multi-Agent (5 agents), Parallel, MCP Protocol |

### Chúc mừng hoàn thành chương trình!

Bạn đã nắm vững:
- **Agent = Model + Harness** — core equation
- **5-Layer Production Harness** — Tool Orchestration → Verification → Context → Guardrails → Observability
- **12-Metric Evaluation Framework** — Retrieval, Generation, Agent, Production
- **Production Deployment** — Docker, CI/CD, Monitoring, Scaling
- **EU AI Act & Governance** — High-Risk AI, RACI, Model Card
- **Multi-Agent Architecture** — Supervisor pattern, Parallelism, MCP Protocol

**Bước tiếp theo:**
- Apply kiến thức vào project thực tế của bạn
- Đóng góp vào Anthropic MCP ecosystem
- Tham khảo: anthropic.com/research, modelcontextprotocol.io